# 02 – Mamba block from scratch

We implement a Mamba (selective SSM) block in pure PyTorch (`src/mamba.py`) and test it piece by piece:

* shapes and NaN checks for every intermediate tensor,
* a closed-form check of the selective scan,
* **causality**: perturbing input at time *t* must not change outputs before *t*,
* **streaming**: `step()` one timestep at a time must match `forward()` to < 1e-5,
* parameter counts.

**Setup.** *Expected output:* project root and versions.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "font.size": 9})

print("project root:", ROOT)
print("torch", torch.__version__, "| numpy", np.__version__)

project root: D:\Project\Development\Mamba_CubeSat
torch 2.15.0+cpu | numpy 2.4.6


## 1. Seed and a small test configuration
Small sizes keep every check fast on CPU. `B` batch, `L` length, `D` = d_model.
*Expected output:* the chosen sizes.

In [2]:
from src.mamba import MambaBlock, MambaStack, selective_scan, count_parameters
from src.train import set_seed

set_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)
B, L, D, N, K = 2, 64, 16, 8, 4
print(f"B={B}  L={L}  d_model={D}  d_state={N}  d_conv={K}")

B=2  L=64  d_model=16  d_state=8  d_conv=4


## 2. Build one `MambaBlock` and list its parameters
`d_inner = expand × d_model = 32`, `dt_rank = ceil(d_model / 16) = 1`.
*Expected output:* `in_proj (64, 16)`, `conv1d (32, 1, 4)` (depthwise), `x_proj (1 + 2N, 32)`, `dt_proj (32, 1)`, `A_log (32, 8)`, `D (32,)`, `out_proj (16, 32)`.

In [3]:
blk = MambaBlock(D, d_state=N, d_conv=K, expand=2)
E, R = blk.d_inner, blk.dt_rank
print(f"d_inner={E}  dt_rank={R}")
for name, p in blk.named_parameters():
    print(f"  {name:18s} {tuple(p.shape)}")
assert blk.in_proj.weight.shape == (2 * E, D)
assert blk.conv1d.weight.shape == (E, 1, K)
assert blk.x_proj.weight.shape == (R + 2 * N, E)
assert blk.A_log.shape == (E, N)

d_inner=32  dt_rank=1
  A_log              (32, 8)
  D                  (32,)
  in_proj.weight     (64, 16)
  conv1d.weight      (32, 1, 4)
  conv1d.bias        (32,)
  x_proj.weight      (17, 32)
  dt_proj.weight     (32, 1)
  dt_proj.bias       (32,)
  out_proj.weight    (16, 32)


## 3. Walk through `forward()` manually
Re-run the forward pass line by line with shape prints. This is the place to debug if something breaks.
*Expected output:* the shape after every stage, `A` strictly negative, Δ strictly positive, and the manual result equal to `blk(x)`.

In [4]:
x = torch.randn(B, L, D)
xz = blk.in_proj(x);                         print("in_proj      ", tuple(xz.shape))
xi, z = xz.chunk(2, dim=-1);                 print("x, z         ", tuple(xi.shape), tuple(z.shape))
xc = blk.conv1d(xi.transpose(1, 2));         print("conv1d (pad) ", tuple(xc.shape), "-> keep first L")
xc = torch.nn.functional.silu(xc[..., :L].transpose(1, 2)); print("conv + SiLU  ", tuple(xc.shape))
dt_low, Bm, Cm = torch.split(blk.x_proj(xc), [R, N, N], dim=-1)
print("x_proj -> dt_low, B, C", tuple(dt_low.shape), tuple(Bm.shape), tuple(Cm.shape))
delta = torch.nn.functional.softplus(blk.dt_proj(dt_low)); print("delta        ", tuple(delta.shape))
A = -torch.exp(blk.A_log);                   print("A            ", tuple(A.shape))
y = selective_scan(xc, delta, A, Bm, Cm, blk.D); print("scan y       ", tuple(y.shape))
out_manual = blk.out_proj(y * torch.nn.functional.silu(z)); print("out          ", tuple(out_manual.shape))

print(f"A range [{A.min():.2f}, {A.max():.2f}]  delta range [{delta.min():.4f}, {delta.max():.4f}]")
assert (A < 0).all() and (delta > 0).all()
assert torch.allclose(out_manual, blk(x), atol=1e-6)
print("manual == blk(x): OK")

in_proj       (2, 64, 64)
x, z          (2, 64, 32) (2, 64, 32)
conv1d (pad)  (2, 32, 67) -> keep first L
conv + SiLU   (2, 64, 32)
x_proj -> dt_low, B, C (2, 64, 1) (2, 64, 8) (2, 64, 8)
delta         (2, 64, 32)
A             (32, 8)
scan y        (2, 64, 32)
out           (2, 64, 16)
A range [-8.00, -1.00]  delta range [0.0014, 0.0885]
manual == blk(x): OK


## 4. Selective scan against a closed form
With one channel, one state, constant Δ, `A = -1`, `B = C = 1`, `u = 1`, `D = 0`, the recursion `h_t = a·h_{t-1} + Δ` (with `a = e^{-Δ}`) has the closed form `h_t = Δ (1 − a^{t+1}) / (1 − a)`.
*Expected output:* max error ≈ 1e-7.

In [5]:
Lc, dlt = 50, 0.1
u = torch.ones(1, Lc, 1, dtype=torch.float64)
delta_c = torch.full((1, Lc, 1), dlt, dtype=torch.float64)
A_c = -torch.ones(1, 1, dtype=torch.float64)
B_c = torch.ones(1, Lc, 1, dtype=torch.float64)
C_c = torch.ones(1, Lc, 1, dtype=torch.float64)
y_c = selective_scan(u, delta_c, A_c, B_c, C_c, torch.zeros(1, dtype=torch.float64))[0, :, 0]
a = np.exp(-dlt)
expected = dlt * (1 - a ** (np.arange(Lc) + 1)) / (1 - a)
err = np.abs(y_c.numpy() - expected).max()
print("max |scan - closed form| =", err)
assert err < 1e-10

max |scan - closed form| = 4.440892098500626e-16


## 5. Full forward: shape and NaN check
*Expected output:* `(2, 64, 16)` and no NaN / Inf.

In [6]:
out = blk(x)
print("output:", tuple(out.shape))
assert out.shape == (B, L, D)
assert torch.isfinite(out).all()
print("finite: OK")

output: (2, 64, 16)
finite: OK


## 6. Residual + LayerNorm stack
`MambaStack` = N × [`x + Mamba(LayerNorm(x))`] followed by a final LayerNorm.
*Expected output:* the module tree and output shape `(2, 64, 16)`.

In [7]:
stack = MambaStack(D, n_layers=3, d_state=N, d_conv=K)
print(stack)
y_stack = stack(x)
print("stack output:", tuple(y_stack.shape))
assert y_stack.shape == (B, L, D) and torch.isfinite(y_stack).all()

MambaStack(
  (layers): ModuleList(
    (0-2): 3 x ResidualMambaLayer(
      (norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
      (mixer): MambaBlock(
        (in_proj): Linear(in_features=16, out_features=64, bias=False)
        (conv1d): Conv1d(32, 32, kernel_size=(4,), stride=(1,), padding=(3,), groups=32)
        (x_proj): Linear(in_features=32, out_features=17, bias=False)
        (dt_proj): Linear(in_features=1, out_features=32, bias=True)
        (out_proj): Linear(in_features=32, out_features=16, bias=False)
      )
    )
  )
  (norm_f): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
)
stack output: (2, 64, 16)


## 7. Causality check (perturbation)
Change the input **only at time t0** and compare outputs. Outputs at times < t0 must be bit-for-bit unchanged; outputs at ≥ t0 should change.
*Expected output:* `max diff before t0 = 0.0`, a clearly non-zero diff from t0 on.

In [8]:
stack.eval()
t0 = 30
x_pert = x.clone()
x_pert[:, t0] += 5.0 * torch.randn(B, D)
with torch.no_grad():
    y0, y1 = stack(x), stack(x_pert)
diff = (y1 - y0).abs().amax(dim=(0, 2))            # per time step
print(f"max diff before t0: {diff[:t0].max().item():.3e}")
print(f"max diff at/after t0: {diff[t0:].max().item():.3e}")
print("diff around t0:", diff[t0 - 3:t0 + 4].numpy().round(4))
assert diff[:t0].max() == 0.0, "information leaked from the future!"
assert diff[t0] > 1e-3

max diff before t0: 0.000e+00
max diff at/after t0: 3.323e+00
diff around t0: [0.     0.     0.     3.3231 0.0801 0.0801 0.0929]


## 7b. Causality check (gradients)
A stricter check: the Jacobian ∂y_t / ∂x_s must be exactly zero for every s > t. We test all output times of a short sequence.
*Expected output:* `max |grad| from the future = 0.0`.

In [9]:
Lg = 16
xg = torch.randn(1, Lg, D, requires_grad=True)
w = torch.randn(D)       # random projection: y.sum() is useless after LayerNorm (sum is constant)
leak = 0.0
for t in range(Lg):
    g, = torch.autograd.grad((stack(xg)[0, t] * w).sum(), xg)
    leak = max(leak, g[0, t + 1:].abs().max().item() if t + 1 < Lg else 0.0)
    assert g[0, : t + 1].abs().sum() > 0          # it does depend on the past
print("max |grad| from the future:", leak)
assert leak == 0.0

max |grad| from the future: 0.0


## 8. Streaming `step()` vs full `forward()`
`step(x_t, conv_state, h)` keeps the last `d_conv` conv inputs and the SSM state `h`. Running it for t = 0..L-1 must reproduce `forward()`.
*Expected output:* max abs diff < 1e-5 in float32 (typically ~1e-7), and ~1e-15 in float64.

In [10]:
@torch.no_grad()
def stream(model, x):
    states = model.init_state(x.shape[0], dtype=x.dtype)
    ys = []
    for t in range(x.shape[1]):
        y_t, states = model.step(x[:, t], states)
        ys.append(y_t)
    return torch.stack(ys, 1)

with torch.no_grad():
    y_full = stack(x)
y_step = stream(stack, x)
d32 = (y_full - y_step).abs().max().item()
print("states per layer: conv", tuple(stack.init_state(B)[0][0].shape), "h", tuple(stack.init_state(B)[0][1].shape))
print(f"float32: max |forward - step| = {d32:.2e}")
assert y_step.shape == y_full.shape and d32 < 1e-5

stack64 = MambaStack(D, n_layers=3, d_state=N, d_conv=K).double().eval()
x64 = x.double()
with torch.no_grad():
    d64 = (stack64(x64) - stream(stack64, x64)).abs().max().item()
print(f"float64: max |forward - step| = {d64:.2e}")
assert d64 < 1e-12

states per layer: conv (2, 32, 4) h (2, 32, 8)
float32: max |forward - step| = 4.77e-07
float64: max |forward - step| = 8.88e-16


Same check on a single `MambaBlock` (no residual / norm), to localise errors if the stack test ever fails.
*Expected output:* < 1e-5.

In [11]:
blk.eval()
conv_state, h = blk.init_state(B)
outs = []
with torch.no_grad():
    for t in range(L):
        o, conv_state, h = blk.step(x[:, t], conv_state, h)
        outs.append(o)
    d_blk = (torch.stack(outs, 1) - blk(x)).abs().max().item()
print(f"MambaBlock: max |forward - step| = {d_blk:.2e}")
assert d_blk < 1e-5

MambaBlock: max |forward - step| = 3.73e-08


## 9. Parameter count
Per block (bias-free projections, as in the reference implementation):
`in_proj 2ED + conv E(K+1) + x_proj E(R+2N) + dt_proj (RE+E) + A_log EN + D E + out_proj DE`.
We also count the full forecaster used in notebook 03 (17 inputs → d_model 32, 2 layers, d_state 16 → 7 outputs).
*Expected output:* the formula matches the module count; the forecaster has ≈ 21 k parameters (embed 576, backbone 20 032, head 231).

In [12]:
def block_params(D, N, K, expand=2):
    E = expand * D; R = int(np.ceil(D / 16))
    return 2 * E * D + E * (K + 1) + E * (R + 2 * N) + (R * E + E) + E * N + E + D * E

print("MambaBlock(d=16, N=8):", count_parameters(blk), "| formula:", block_params(D, N, K))
assert count_parameters(blk) == block_params(D, N, K)

from src.models import MambaForecaster
fc = MambaForecaster(in_channels=17, out_channels=7, d_model=32, n_layers=2, d_state=16)
for name, m in [("embed", fc.embed), ("backbone", fc.backbone), ("head", fc.head)]:
    print(f"  {name:9s} {count_parameters(m):7,d}")
print(f"  {'total':9s} {count_parameters(fc):7,d}")
assert count_parameters(fc.backbone.layers[0].mixer) == block_params(32, 16, 4)

MambaBlock(d=16, N=8): 2592 | formula: 2592
  embed         576
  backbone   20,032
  head          231
  total      20,839


## 10. CPU timing
The explicit for-loop scan is easy to read but sequential in L. Time one forward + backward with the training sizes (B=32, L=512) to know what to expect in notebook 03.
*Expected output:* roughly 1–5 s per batch on a laptop CPU (the first call is slower: allocator warm-up). The scan uses `unbind(1)` rather than `dA[:, t]` inside the loop — indexing would make the backward pass O(L²) and ~30× slower.

In [13]:
import time
xb = torch.randn(32, 511, 17)
fc.train()
t0 = time.time()
loss = fc(xb).pow(2).mean(); loss.backward()
print(f"forward+backward B=32 L=511: {time.time() - t0:.2f} s")

forward+backward B=32 L=511: 6.57 s


## Summary
* `MambaBlock` = in_proj → causal depthwise conv → SiLU → input-dependent (Δ, B, C) → selective scan with `A = −exp(A_log)` and D skip → SiLU(z) gate → out_proj.
* The scan matches the closed form; the stack is strictly causal (perturbation **and** gradient tests); streaming `step()` matches `forward()`.
* Next: `03_train.ipynb` trains the forecaster for next-step prediction and evaluates anomaly detection.